# Linear & Quadratic Discriminant Analysis

_Classical ML_

**Fit a Gaussian per class. Share covariance ⇒ line. Per-class ⇒ curve.**

Model each class as its own Gaussian bell. To classify a point, ask which bell explains it better.

     If all classes are forced to share one covariance (one shape), the boundary between them is a straight line: that is LDA (Linear Discriminant Analysis).

---

This notebook is a practice scaffold for the **Linear & Quadratic Discriminant Analysis** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

### Step 1 — Make a two-class, two-feature dataset

To compare LDA and QDA we need a labelled set we can also picture. We synthesise 400 points with two informative features and a clear gap between the classes, then hold out a quarter as a test set so we can measure accuracy on points the models never trained on.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=400, n_features=2, n_redundant=0,
                           n_informative=2, n_clusters_per_class=1,
                           class_sep=1.2, random_state=0)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)

### Step 2 — Fit LDA and QDA

Both methods model each class as a Gaussian bell. LDA forces every class to share one covariance (one bell shape), which makes the boundary a straight line. QDA lets each class keep its own covariance, so its boundary can curve. We fit both on the same training data.

In [ ]:
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis)

lda = LinearDiscriminantAnalysis().fit(Xtr, ytr)
qda = QuadraticDiscriminantAnalysis().fit(Xtr, ytr)

### Step 3 — Compare accuracy and inspect predictions

The class priors are just the fraction of training points in each class — LDA uses them to weigh the bells. We then score both models on the held-out test set and peek at the first few predictions next to the true labels to see them agree.

In [ ]:
lda_accuracy = lda.score(Xte, yte)
qda_accuracy = qda.score(Xte, yte)

print("class priors (LDA):", np.round(lda.priors_, 3))
print("LDA test accuracy:", round(lda_accuracy, 3))
print("QDA test accuracy:", round(qda_accuracy, 3))
print("LDA predicts (first 8):", lda.predict(Xte[:8]))
print("true labels  (first 8):", yte[:8])

## Visualize the data & results

_Splitting two Wine cultivars by flavanoids and color intensity: straight LDA line or curved QDA boundary?_

### Step 1 — Pick two Wine cultivars and two features

To draw the boundaries we drop to two dimensions on real data. We keep only cultivars 0 and 1 and describe each wine by two columns — flavanoids and color intensity — then fit both LDA and QDA on these two features.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis)

wine = load_wine()
mask = wine.target < 2                 # keep cultivars 0 and 1
X = wine.data[mask][:, [6, 9]]         # columns: flavanoids, color_intensity
y = wine.target[mask]

lda = LinearDiscriminantAnalysis().fit(X, y)
qda = QuadraticDiscriminantAnalysis().fit(X, y)

### Step 2 — Build a grid over the feature plane

To draw a decision boundary we evaluate each model at every point of a fine grid covering the data, then trace the line where its prediction flips from one class to the other. `meshgrid` lays out the grid and we flatten it into a list of points the models can score.

In [ ]:
# A 300x300 grid spanning the data, with a little margin on each side.
gx = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300)
gy = np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300)
XX, YY = np.meshgrid(gx, gy)

# Flatten the grid into one (x, y) point per row.
grid = np.c_[XX.ravel(), YY.ravel()]

### Step 3 — Plot the points and both boundaries

We scatter the wines coloured by cultivar, then overlay each model's boundary as the contour where its prediction switches at 0.5. The LDA boundary comes out straight; the QDA boundary (dashed) is allowed to curve because each class keeps its own covariance.

In [ ]:
colors = np.array(["#4ea1ff", "#7ee787"])
plt.scatter(X[:, 0], X[:, 1], c=colors[y], s=18)

# LDA: solid contour where the predicted class flips.
lda_grid = lda.predict(grid).reshape(XX.shape)
plt.contour(XX, YY, lda_grid, levels=[0.5], colors="#ffb454")

# QDA: dashed contour, free to curve.
qda_grid = qda.predict(grid).reshape(XX.shape)
plt.contour(XX, YY, qda_grid, levels=[0.5], colors="#c89bff", linestyles="dashed")

plt.title("LDA line vs QDA curve on Wine (cultivars 0 and 1)")
plt.xlabel("flavanoids")
plt.ylabel("color intensity")
plt.show()